In [1]:
!pip install pandas
!pip install openpyxl

!pip install chromadb sentence-transformers


In [2]:
import pandas as pd

def load_questionnaire_sheet(file_path: str, sheet_name="Questionnaire"):

    df = pd.read_excel(file_path, sheet_name=sheet_name, header=None)
    df = df.dropna(how="all").reset_index(drop=True)

    df_lower = df.apply(lambda col: col.astype(str).str.strip().str.lower())

    header_mask = df_lower.apply(
        lambda r: set(["field","description","value"]).issubset(set(r)),
        axis=1
    )

    if not header_mask.any():
        raise ValueError("Header row (field, description, value) not found.")

    header_idx = header_mask.idxmax()

    df.columns = (
        df.iloc[header_idx]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    df = df.iloc[header_idx + 1:].reset_index(drop=True)

    expected_cols = {"field", "description", "value"}
    if not expected_cols.issubset(set(df.columns)):
        raise ValueError("Unexpected column structure in Questionnaire sheet.")

    return df


def split_questionnaire_sections(df: pd.DataFrame):

    category_rows = df[
        df["field"].astype(str).str.strip().str.lower() == "category"
    ]

    if category_rows.empty:
        raise ValueError("Questionnaire section header not found.")

    split_idx = category_rows.index[0]

    project_df = df.iloc[:split_idx].copy().reset_index(drop=True)
    questionnaire_df = df.iloc[split_idx + 1:].copy().reset_index(drop=True)

    questionnaire_df.columns = ["category", "description", "response"]

    questionnaire_df = questionnaire_df.apply(
        lambda col: col.map(lambda x: x.strip() if isinstance(x, str) else x)
    )

    return project_df, questionnaire_df

In [4]:
# testing
file_path="Updated_DataLake_Questionnaire.xlsx"
df = load_questionnaire_sheet(file_path)
project_df, questionnaire_df = split_questionnaire_sections(df)

print(project_df.shape)
print(questionnaire_df.shape)
print(questionnaire_df.columns.tolist())
print(project_df.columns.tolist())

(5, 3)
(52, 3)
['category', 'description', 'response']
['field', 'description', 'value']


In [5]:
# Resons of doing  this
#Control Layer for Routing
# Metadata Enrichment for All Chunks
# Domain Isolation
# Zero-Hallucination Governance Answers
# This dictionary is NOT for chunking.
# It is for:
# Control layer
# Routing
# Metadata filtering
# Direct governance Q&A

In [6]:
# USED :MINIMAL CONTROL METADATA 
# Extracted governanace attribute needed for : ROUTING , FILTERING,DETERMINISTIC ANSWERS
# gOVERNANACE GETS SPLITS into
# 1 CONTROL LAYER :used for domain routing, matadata  filtering,direct Q&A :stored in global_governanace_content
# 2 CONTENT LAYER :semantic search ,risk queries,vendor queries,conatct queries,chnage mangemnet queries-stored in questionire_df

In [7]:
# Metadata 
import re

def normalize_key(text: str) -> str:
    text = text.strip().lower()
    text = re.sub(r"[\/\s]+", "_", text)   # replace / and spaces with _
    text = re.sub(r"_+", "_", text)        # collapse multiple underscores
    return text

def extract_project_metadata(project_df: pd.DataFrame):
    
    metadata = {}
    
    for _, row in project_df.iterrows():
        raw_key = str(row["field"])
        key = normalize_key(raw_key)
        value = row["value"]
        
        if pd.notna(value):
            metadata[key] = str(value).strip()
    
    return metadata


project_metadata = extract_project_metadata(project_df)

print(project_metadata)

{'project_name': 'MBP 3.11 Upgrade', 'planview_project_id': '8679', 'impact_score': '90', 'strategic_initiative_program_name': 'EDP', 'enactment_pod_name': 'Ancillary Integration'}


In [8]:
def extract_governance_attributes(questionnaire_df: pd.DataFrame):
    
    governance_meta = {}
    
    # Normalize columns
    questionnaire_df["category"] = questionnaire_df["category"].str.strip()
    questionnaire_df["description"] = questionnaire_df["description"].str.strip()
    
    for _, row in questionnaire_df.iterrows():
        category = str(row["category"]).strip().lower()
        description = str(row["description"]).strip().lower()
        response = row["response"]
        
        if pd.isna(response):
            continue
        
        response = str(response).strip()
        
        # Extract based on known signals
        
        if category == "source name":
            governance_meta["source_name"] = response
        
        if category == "business line":
            governance_meta["business_line"] = response
        
        if category == "data access" and "host type" in description:
            governance_meta["host_type"] = response
        
        if category == "data access" and "delivery method" in description:
            governance_meta["delivery_method"] = response
        
        if category == "availability" and "how often" in description:
            governance_meta["data_frequency"] = response
        
        if category == "file/ db information" and "security" in description.lower():
            governance_meta["security_classification"] = response
        
        if category == "environments":
            governance_meta["environments"] = response
        
        if category == "data retention period":
            governance_meta["data_retention_period"] = response
        
        if category == "source owner approval":
            governance_meta["source_owner_approval"] = response
    
    return governance_meta


governance_attributes = extract_governance_attributes(questionnaire_df)

print(governance_attributes)

{'source_name': 'FIS Profile', 'business_line': 'Consumer', 'host_type': 'External', 'delivery_method': 'GIS', 'data_frequency': 'Daily', 'security_classification': 'NPPI', 'environments': 'PROD,QA', 'data_retention_period': 'Default 7 year', 'source_owner_approval': 'Pending approval from Source Owner'}


In [9]:
global_governance_context = {
    **project_metadata,
    **governance_attributes
}

print(global_governance_context)

{'project_name': 'MBP 3.11 Upgrade', 'planview_project_id': '8679', 'impact_score': '90', 'strategic_initiative_program_name': 'EDP', 'enactment_pod_name': 'Ancillary Integration', 'source_name': 'FIS Profile', 'business_line': 'Consumer', 'host_type': 'External', 'delivery_method': 'GIS', 'data_frequency': 'Daily', 'security_classification': 'NPPI', 'environments': 'PROD,QA', 'data_retention_period': 'Default 7 year', 'source_owner_approval': 'Pending approval from Source Owner'}


In [10]:
# # File information Sheet :loading,cleaning,Normalise columns and 
# # file names,validate uniqueness,prepare for enrichment


In [11]:
import pandas as pd
import re



 # Detect header row dynamically

def detect_header_row(df: pd.DataFrame):

    df_lower = df.apply(lambda col: col.astype(str).str.strip().str.lower())

    header_mask = df_lower.apply(
        lambda r: "file name" in list(r),
        axis=1
    )

    if not header_mask.any():
        raise ValueError("Header row containing 'File Name' not found.")

    return header_mask.idxmax()


#  Normalize column names
def normalize_columns(df: pd.DataFrame):

    df = df.copy()

    def clean_col(col):
        col = str(col).strip().lower()
        col = col.replace("\n", " ")
        col = re.sub(r"[^\w\s]", "", col)
        col = re.sub(r"\s+", "_", col)
        col = re.sub(r"_+", "_", col)
        return col.strip("_")

    df.columns = [clean_col(c) for c in df.columns]
    return df



# Build standardized file key
# (used to link Sheet 2 and Sheet 3)

def build_standardized_file_key(filename: str):

    if pd.isna(filename):
        return None

    filename = str(filename).lower().strip()

    # Remove extension
    filename = re.sub(r"\.txt$", "", filename)
    filename = re.sub(r"\.csv$", "", filename)

    # Normalize separators
    filename = filename.replace("-", "_")

    # Remove only trailing date patterns
    filename = re.sub(r"_\d{8}$", "", filename)
    filename = re.sub(r"_\d{4}_\d{2}_\d{2}$", "", filename)
    filename = re.sub(r"_yyyymmdd$", "", filename)
    filename = re.sub(r"_yyyy_mm_dd$", "", filename)

    # Clean multiple underscores
    filename = re.sub(r"_+", "_", filename)
    filename = filename.strip("_")

    return filename


# Main loader function

def load_file_information_sheet(
    file_path: str,
    sheet_name="File Information if File Based"
):

    # Load raw
    raw_df = pd.read_excel(file_path, sheet_name=sheet_name, header=None)
    raw_df = raw_df.dropna(how="all").reset_index(drop=True)

    # Detect header row dynamically
    header_idx = detect_header_row(raw_df)

    df = raw_df.iloc[header_idx:].reset_index(drop=True)
    df.columns = df.iloc[0]
    df = df.iloc[1:].reset_index(drop=True)

    # Normalize column names
    df = normalize_columns(df)

    # Validate required columns
    required_columns = [
        "file_name",
        "transmission_type_gisndmother",
        "frequency_dailyweeklymonthly",
        "fulldelta",
        "datalake_schema_name_raw_cl"
    ]

    for col in required_columns:
        if col not in df.columns:
            raise ValueError(f"Required column missing: {col}")

    # Build standardized key for linking
    df["standardized_file_key"] = (
        df["file_name"]
        .apply(build_standardized_file_key)
    )

    # Validate key generation
    if df["standardized_file_key"].isnull().any():
        raise ValueError("Some standardized file keys could not be generated.")

    if df["standardized_file_key"].duplicated().any():
        raise ValueError("Duplicate standardized file keys detected.")

    # Final validation: expected no. of files
    if len(df) == 0:
        raise ValueError("File Information sheet is empty.")

    return df.reset_index(drop=True)

In [12]:
file_information_df = load_file_information_sheet(file_path)

print("Shape:", file_information_df.shape)
print("\nFile keys:")
print(file_information_df[["file_name", "standardized_file_key"]])
print("\nTotal files:", len(file_information_df))

Shape: (13, 26)

File keys:
                                            file_name  \
0                MBP_AccountExtract_4300_YYYYMMDD.txt   
1   MBP_arrangement_extract_delta_4300_YYYY-MM-DD.txt   
2   MBP_ext_att_ar_instance_extract_delta_4300_YYY...   
3   MBP_ext_att_ar_instance_extract_delta_totals_4...   
4     MBP_ext_att_def_extract_delta_4300_YYYYMMDD.txt   
5   MBP_ext_att_def_extract_delta_Totals_4300_yyyy...   
6   MBP_ext_att_ip_instance_extract_delta_4300_YYY...   
7   MBP_ext_att_ip_instance_extract_delta_totals_4...   
8   MBP_involved_party_extract_delta_4300_YYYY-MM-...   
9   MBP_involved_party_extract_delta_totals_4300_Y...   
10  MBP_ip_ar_rltnp_extract_delta_4300_YYYY-MM-DD.txt   
11  MBP_ip_ip_rltnp_extract_delta_4300_YYYY-MM-DD.txt   
12           MBP_TransactionExtract_4300_YYYYMMDD.txt   

                                standardized_file_key  
0                             mbp_accountextract_4300  
1                  mbp_arrangement_extract_delta_4300  
2    

In [13]:
# standardize file key will be used to linked sheet 2 with sheet 3

In [14]:
# Sheet 1: sturctured goverance metadtaa 
        # control layer dictonary 
        # questionanire content preseved for chunking  
#sheet 2:clean injestion metadata 
        # validate required columns  
        # No duplicate file keys 
        # stable linking key created 

# why this is useful 
# When user asks:
# What is the frequency of MBP_AccountExtract?
# System flow will be:
# Extract file name
# Convert to standardized_file_key
# Filter file_information domain by that key
# Return frequency
# No semantic search required.


In [15]:
# loading 3rd sheet 

# SHEET 3 — Metadata Mapping Loader


def load_metadata_mapping_sheet(
    file_path: str,
    file_information_df: pd.DataFrame,
    sheet_name="Metadata_OR_Mapping"
):

    # Load raw
    raw_df = pd.read_excel(file_path, sheet_name=sheet_name, header=None)
    raw_df = raw_df.dropna(how="all").reset_index(drop=True)

    # Detect header row dynamically
    df_lower = raw_df.apply(lambda col: col.astype(str).str.strip().str.lower())

    header_mask = df_lower.apply(
        lambda r: "actual file name" in list(r),
        axis=1
    )

    if not header_mask.any():
        raise ValueError("Header row containing 'Actual File Name' not found.")

    header_idx = header_mask.idxmax()

    df = raw_df.iloc[header_idx:].reset_index(drop=True)
    df.columns = df.iloc[0]
    df = df.iloc[1:].reset_index(drop=True)

    # Normalize column names
    df = normalize_columns(df)

    # Validate required columns
    required_columns = [
        "actual_file_name",
        "record_type",
        "schema_name",
        "table_name",
        "column_name"
    ]

    for col in required_columns:
        if col not in df.columns:
            raise ValueError(f"Required column missing in Sheet 3: {col}")

    # Build standardized file key
    df["standardized_file_key"] = (
        df["actual_file_name"]
        .apply(build_standardized_file_key)
    )

    if df["standardized_file_key"].isnull().any():
        raise ValueError("Some standardized file keys could not be generated in Sheet 3.")

    # Validate alignment with Sheet 2
    sheet2_keys = set(file_information_df["standardized_file_key"])
    sheet3_keys = set(df["standardized_file_key"])

    missing_keys = sheet3_keys - sheet2_keys

    if missing_keys:
        raise ValueError(
            f"Sheet 3 contains files not present in Sheet 2: {missing_keys}"
        )

    return df.reset_index(drop=True)

In [16]:
# testing 
metadata_mapping_df = load_metadata_mapping_sheet(
    file_path,
    file_information_df
)

print("Shape:", metadata_mapping_df.shape)
print("Unique files:", metadata_mapping_df["standardized_file_key"].nunique())

Shape: (64, 14)
Unique files: 13


In [17]:
# merging sheet 2 and sheet 3 using standardized_file_key  
# Group sheet 3 by :
        # standardize filekey 
        # record type 
        # table name
# Attach file-level metadtaa fr sheet 2
#create internal file-tree structure  


In [18]:
# building a hierarchical in-memory structure that merges:
# Sheet 2 → file-level metadata
# Sheet 3 → record-type + table + fields

In [19]:
# With hierarchy:
# Each chunk will represent:
# One file
# + One record type
# + One table
# + List of its fields
# + File-level metadata attached


In [20]:
def build_file_tree(file_information_df, metadata_mapping_df):

    file_tree = {}

    # Build base structure from Sheet 2
    for _, row in file_information_df.iterrows():

        key = row["standardized_file_key"]

        file_tree[key] = {
            "file_info": row.to_dict(),
            "mappings": {}
        }

    # Populate mapping layer from Sheet 3
    for _, row in metadata_mapping_df.iterrows():

        key = row["standardized_file_key"]
        record_type = str(row["record_type"])
        table_name = row["table_name"]

        if key not in file_tree:
            continue

        if record_type not in file_tree[key]["mappings"]:
            file_tree[key]["mappings"][record_type] = {}

        if table_name not in file_tree[key]["mappings"][record_type]:
            file_tree[key]["mappings"][record_type][table_name] = []

        file_tree[key]["mappings"][record_type][table_name].append(row.to_dict())

    return file_tree

In [21]:
# testing 
file_tree = build_file_tree(file_information_df, metadata_mapping_df)

print("Total files in tree:", len(file_tree))

first_key = list(file_tree.keys())[0]
print("\nExample file key:", first_key)

print("\nRecord types under first file:")
print(file_tree[first_key]["mappings"].keys())

Total files in tree: 13

Example file key: mbp_accountextract_4300

Record types under first file:
dict_keys(['9000'])


In [22]:
# it means : 13 file levele parent nodes 
# each file represents one physical ingestion file frm sheet 2 


In [23]:
# this is the file tree structure which is build inside memory
# file_tree = {
#     standardized_file_key: {
#         "file_info": {...},          # from Sheet 2
#         "mappings": {
#             record_type: {
                # table_name: [field rows]         from sheet 3 
#             }
#         } 
#     }
# } (what we are creting )

In [24]:
# Example: file_tree["mbp_accountextract_4300"] 
# {
#     "file_info": {
#         "file_name": "MBP_AccountExtract_4300_YYYYMMDD.txt",
#         "transmission_type_gisndmother": "GIS",
#         "frequency_dailyweeklymonthly": "Daily",
#         "fulldelta": "Delta",
#         "autosys_start": "3 AM EST",
#         "datalake_schema_name_raw_cl": "DEPOSITS_TERRANOVA",
#         ...
#     },

#     "mappings": {

#         "9000": {

#             "APD_AR_TDL_DP": [

#                 {
#                     "field_name": "ARID_ITEM_IBAN",
#                     "field_description": "Arrangement Account IBAN Number",
#                     "field_type": "STRING",
#                     "max_length": 40,
#                     "column_name": "ARID_ITEM_IBAN",
#                     "datatype": "STRING"
#                 },

#                 {
#                     "field_name": "REC_TYP",
#                     "field_description": "Record Type",
#                     "field_type": "NUMBER",
#                     "max_length": 4,
#                     "column_name": "REC_TYP",
#                     "datatype": "NUMBER"
#                 },

#                 {
#                     "field_name": "ARF_IDFR",
#                     "field_description": "Arrangement Identifier",
#                     "field_type": "NUMBER",
#                     "max_length": 15,
#                     "column_name": "ARF_IDFR",
#                     "datatype": "NUMBER"
#                 },

#                 ...
#             ]
#         }
#     }
# }


In [25]:
# purpose: grouping sheet 3 rows + attch sheet2 info: to create the sturcted chunk 
# it exist during chunk creation 
# not stored in chroma , not used in retrival 
# see function :build_technical_chunks (donw chunking for sheet3 using file_tree)

In [26]:
print("Sheet 2 files:", len(file_information_df))
print("Tree files:", len(file_tree))
sheet2_keys = set(file_information_df["standardized_file_key"])
tree_keys = set(file_tree.keys())

print("Missing in tree:", sheet2_keys - tree_keys)

sheet3_keys = set(metadata_mapping_df["standardized_file_key"])

print("Sheet3 not in Sheet2:", sheet3_keys - sheet2_keys)

key = "mbp_accountextract_4300"

print("Record types:", file_tree[key]["mappings"].keys())

record_type = "9000"
table_name = "APD_AR_TDL_DP"

fields = file_tree[key]["mappings"][record_type][table_name]

print("Number of fields:", len(fields))
print("Sample field:", fields[0])

Sheet 2 files: 13
Tree files: 13
Missing in tree: set()
Sheet3 not in Sheet2: set()
Record types: dict_keys(['9000'])
Number of fields: 5
Sample field: {'file_no': 1, 'record_type': 9000, 'actual_file_name': 'MBP_AccountExtract_4300_YYYYMMDD.txt', 'field_name': 'ARID_ITEM_IBAN', 'field_description': 'Arrangement Account IBAN Number', 'field_type': 'STRING', 'max_length': 40, 'connectiontype': 'GIS', 'schema_name': 'CL_DEPOSITS_TERRANOVA', 'table_name': 'APD_AR_TDL_DP', 'column_name': 'ARID_ITEM_IBAN', 'datatype': 'STRING', 'standardized_file_key': 'mbp_accountextract_4300'}


In [27]:
raw_count = metadata_mapping_df[
    metadata_mapping_df["standardized_file_key"] == key
].shape[0]

tree_count = sum(
    len(fields)
    for rt in file_tree[key]["mappings"].values()
    for fields in rt.values()
)

print("Raw rows:", raw_count)
print("Tree rows:", tree_count)

Raw rows: 5
Tree rows: 5


In [28]:
# # why sshet1 is not inside file_tree:
# # global governance metadata 
# it applies to entire source , all 13 files,all record types, all tables, 
# it is not file-specific.  



# sheet 1: serves two purpose :
# 1)governanace chunking (content layer) 
# What are the risks?
# Who is the vendor?
# What is the retention period?
# Who are the contacts?
# What is the sourcing strategy?

# 2) Meatdata control layerr 
# That is used for:
# domain routing
# Deterministic answers
# Filtering
# Future multi-source isolation
# That dictionary is NOT embedded.
# It is control metadata.



    

In [29]:
# final structure will be : 
# 1) Goveranace chunks 
# domain==governance  
# no file key  
# used for governace que and answer   

# 2)File informaton chunks
# Domain ==file information  
# one chunk per  andfilee  


# 3) Technical mapping (sheet 3 via file_tree)  
# domain =technical mappingg   
# one chunk per :file+record type   



In [30]:
# sheet 1:
# chunk content :
    
# Project Name: MBP 3.11 Upgrade
# Source Name: FIS Profile
# Business Line: Consumer
# Host Type: External
# Delivery Method: GIS
# Data Frequency: Daily
# Security Classification: NPPI
# Environments: PROD, QA
# Retention Period: Default 7 year
# Source Owner Approval: Pending approval from Source Owner


# Metadata:

# {
#   domain: "governance",
#   source_name: "FIS Profile",
#   project_name: "MBP 3.11 Upgrade"
# }



    


In [31]:
# # File information chunk should look lik e
# Example for mbp_accountextract_4300:

# File Name: MBP_AccountExtract_4300_YYYYMMDD.txt
# Description: CAX Account File
# Transmission Type: GIS
# Frequency: Daily
# Load Type: Delta
# Autosys Start: 3 AM EST
# Schema: DEPOSITS_TERRANOVA
# Encrypted: N
# Compressed: N
# Producer Mailbox: DTECTB_FISGLOBAL1
# Consumer Mailbox: DTI51171_ADVANCANAL02

# Metadata:

# {
#   domain: "file_information",
#   file_key: "mbp_accountextract_4300",
#   frequency: "Daily",
#   load_type: "Delta",
#   transmission: "GIS",
#   schema: "DEPOSITS_TERRANOVA"
# }


In [32]:

# # Technical mapping chunks 
# one chunk per :  file + record_type + table   

# example :
# File: MBP_AccountExtract_4300_YYYYMMDD.txt
# Record Type: 9000
# Target Schema: CL_DEPOSITS_TERRANOVA
# Target Table: APD_AR_TDL_DP
# Frequency: Daily
# Transmission: GIS
# Load Type: Delta

# Fields:
# - ARID_ITEM_IBAN (STRING 40) → ARID_ITEM_IBAN (STRING 40)
# - REC_TYP (NUMBER 4) → REC_TYP (NUMBER 4)
# - ARF_IDFR (NUMBER 15) → ARF_IDFR (NUMBER 15)
# - ARID_ITEM (STRING 40) → ARID_ITEM (STRING 40)
# - ARID_ITEM_MICR (STRING 40) → ARID_ITEM_MICR (STRING 40)

# metadata:    


# {
#   domain: "technical_mapping",
#   file_key: "mbp_accountextract_4300",
#   record_type: "9000",
#   schema: "CL_DEPOSITS_TERRANOVA",
#   table: "APD_AR_TDL_DP",
#   frequency: "Daily",
#   load_type: "Delta",
#   transmission: "GIS"
# }


In [33]:
# Building technical chunks builder  

def build_technical_chunks(file_tree):

    chunks = []

    for file_key, file_data in file_tree.items():

        file_info = file_data["file_info"]
        mappings = file_data["mappings"]

        for record_type, tables in mappings.items():

            for table_name, fields in tables.items():

                # -------- Content --------
                content_lines = []

                content_lines.append(
                    f"File: {file_info.get('file_name')}"
                )
                content_lines.append(
                    f"Record Type: {record_type}"
                )
                content_lines.append(
                    f"Target Schema: {fields[0].get('schema_name')}"
                )
                content_lines.append(
                    f"Target Table: {table_name}"
                )

                # Enriched metadata from Sheet 2
                content_lines.append(
                    f"Frequency: {file_info.get('frequency_dailyweeklymonthly')}"
                )
                content_lines.append(
                    f"Transmission: {file_info.get('transmission_type_gisndmother')}"
                )
                content_lines.append(
                    f"Load Type: {file_info.get('fulldelta')}"
                )

                content_lines.append("\nFields:")

                for field in fields:
                    content_lines.append(
                        f"- {field.get('field_name')} "
                        f"({field.get('field_type')} {field.get('max_length')}) "
                        f"→ {field.get('column_name')} "
                        f"({field.get('datatype')} {field.get('max_length1')})"
                    )

                content = "\n".join(content_lines)

                # -------- Metadata --------
                metadata = {
                    "domain": "technical_mapping",
                    "file_key": file_key,
                    "record_type": str(record_type),
                    "schema": fields[0].get("schema_name"),
                    "table": table_name,
                    "frequency": file_info.get("frequency_dailyweeklymonthly"),
                    "load_type": file_info.get("fulldelta"),
                    "transmission": file_info.get("transmission_type_gisndmother")
                }

                chunks.append({
                    "content": content,
                    "metadata": metadata
                })

    return chunks

In [34]:
technical_chunks = build_technical_chunks(file_tree)

print("Total technical chunks:", len(technical_chunks))
print("\nSample chunk:\n")
print(technical_chunks[0]["content"])
print("\nMetadata:\n")
print(technical_chunks[0]["metadata"])

Total technical chunks: 13

Sample chunk:

File: MBP_AccountExtract_4300_YYYYMMDD.txt
Record Type: 9000
Target Schema: CL_DEPOSITS_TERRANOVA
Target Table: APD_AR_TDL_DP
Frequency: Daily
Transmission: GIS
Load Type: Delta

Fields:
- ARID_ITEM_IBAN (STRING 40) → ARID_ITEM_IBAN (STRING None)
- REC_TYP (NUMBER 4) → REC_TYP (NUMBER None)
- ARF_IDFR (NUMBER 15) → ARF_IDFR (NUMBER None)
- ARID_ITEM (STRING 40) → ARID_ITEM (STRING None)
- ARID_ITEM_MICR (STRING 40) → ARID_ITEM_MICR (STRING None)

Metadata:

{'domain': 'technical_mapping', 'file_key': 'mbp_accountextract_4300', 'record_type': '9000', 'schema': 'CL_DEPOSITS_TERRANOVA', 'table': 'APD_AR_TDL_DP', 'frequency': 'Daily', 'load_type': 'Delta', 'transmission': 'GIS'}


In [35]:
# eachn chunk in vector db will has :document(content) and metadata(the dictionary) 
# Metadata is stored alongside the embedding.
# It does NOT get mixed inside text.
# It is stored separately.

In [36]:
# how metadta affect retrival  :
# ex: what is fre of mbp_accountextract? 
# system will: detect domain: file_information,extract file_key :mbp_accountextract_4300
# filter vector db using metadata

In [37]:
# sheet2 chunks 
# These answer questions like:
# What is the frequency of MBP_AccountExtract?
# What is the transmission type of arrangement file?
# What is the autosys start time?
# Is the file encrypted?
# What is the schema name?


# This layer is file-level, not field-level.
# One chunk per file.




In [38]:
# chunk format for sheet2:

# chunk content:
    
# File Name: MBP_AccountExtract_4300_YYYYMMDD.txt
# Description: CAX Account File
# Transmission Type: GIS
# Frequency: Daily
# Load Type: Delta
# Autosys Start: 3 AM EST
# Schema: DEPOSITS_TERRANOVA
# Encrypted: N
# Compressed: N
# Producer Mailbox: DTECTB_FISGLOBAL1
# Consumer Mailbox: DTI51171_ADVANCANAL02


# metadataa :

# {
#   domain: "file_information",
#   file_key: "mbp_accountextract_4300",
#   frequency: "Daily",
#   load_type: "Delta",
#   transmission: "GIS",
#   schema: "DEPOSITS_TERRANOVA"
# }
    


In [39]:
#sheet 2 chunks
def build_file_information_chunks(file_information_df):

    chunks = []

    for _, row in file_information_df.iterrows():

        content_lines = []

        content_lines.append(f"File Name: {row.get('file_name')}")
        content_lines.append(f"Description: {row.get('file_description_include_subject_area')}")
        content_lines.append(f"Transmission Type: {row.get('transmission_type_gisndmother')}")
        content_lines.append(f"Frequency: {row.get('frequency_dailyweeklymonthly')}")
        content_lines.append(f"Load Type: {row.get('fulldelta')}")
        content_lines.append(f"Autosys Start: {row.get('autosys_start')}")
        content_lines.append(f"Schema: {row.get('datalake_schema_name_raw_cl')}")
        content_lines.append(f"Encrypted: {row.get('encrypted_yn')}")
        content_lines.append(f"Compressed: {row.get('compressed_yn')}")
        content_lines.append(f"Producer Mailbox: {row.get('original_producer_mailbox')}")
        content_lines.append(f"Consumer Mailbox: {row.get('consumer_mailbox')}")

        content = "\n".join(content_lines)

        metadata = {
            "domain": "file_information",
            "file_key": row.get("standardized_file_key"),
            "frequency": row.get("frequency_dailyweeklymonthly"),
            "load_type": row.get("fulldelta"),
            "transmission": row.get("transmission_type_gisndmother"),
            "schema": row.get("datalake_schema_name_raw_cl")
        }

        chunks.append({
            "content": content,
            "metadata": metadata
        })

    return chunks

In [40]:
# testing

file_chunks = build_file_information_chunks(file_information_df)

print("Total file chunks:", len(file_chunks))
print("\nSample file chunk:\n")
print(file_chunks[0]["content"])
print("\nMetadata:\n")
print(file_chunks[0]["metadata"])


Total file chunks: 13

Sample file chunk:

File Name: MBP_AccountExtract_4300_YYYYMMDD.txt
Description: CAX Account File
Transmission Type: GIS
Frequency: Daily
Load Type: Delta
Autosys Start: 3 AM EST
Schema: DEPOSITS_TERRANOVA
Encrypted: N
Compressed: N
Producer Mailbox: DTECTB_FISGLOBAL1
Consumer Mailbox: DTI51171_ADVANCANAL02

Metadata:

{'domain': 'file_information', 'file_key': 'mbp_accountextract_4300', 'frequency': 'Daily', 'load_type': 'Delta', 'transmission': 'GIS', 'schema': 'DEPOSITS_TERRANOVA'}


In [41]:
# Metadata of sheet 2:

# {'domain': 'file_information', 'file_key': 'mbp_accountextract_4300', 
# 'frequency': 'Daily', 'load_type': 'Delta', 'transmission': 'GIS', 
# 'schema': 'DEPOSITS_TERRANOVA'}

# metadta of sheet 3:

# {'domain': 'technical_mapping', 'file_key': 'mbp_accountextract_4300',
#  'record_type': '9000', 'schema': 'CL_DEPOSITS_TERRANOVA', 'table': 'APD_AR_TDL_DP', 
#  'frequency': 'Daily','load_type': 'Delta', 'transmission': 'GIS'}


# sheet 2 -> one to many realtionship ->sheet 3 
# sheet 2= parent ,sheet3 =children  
# vec DB do not store any relationship 

# what I did: store chunks independetly ,but they share same file_key metadata
# at retrival we filter using that key .
# relationship  : logical and metadata driven ,



In [42]:
def build_governance_chunks(project_df, questionnaire_df, global_governance_context):

    chunks = []

   
    # Project Overview Chunk
    
    project_lines = []

    for _, row in project_df.iterrows():
        if pd.notna(row["value"]):
            project_lines.append(f"{row['field']}: {row['value']}")

    project_content = "\n".join(project_lines)

    project_metadata = {
        "domain": "governance",
        "section": "project_overview",
        "source_name": global_governance_context.get("source_name"),
        "project_name": global_governance_context.get("project_name")
    }

    chunks.append({
        "content": project_content,
        "metadata": project_metadata
    })

    # --------------------------------------------------
    # Questionnaire Chunks (Grouped by Category)
    # --------------------------------------------------
    grouped = questionnaire_df.groupby("category")

    for category, group in grouped:

        lines = []

        for _, row in group.iterrows():
            if pd.notna(row["response"]):
                lines.append(
                    f"{row['description']}: {row['response']}"
                )

        if not lines:
            continue

        content = f"Category: {category}\n\n" + "\n".join(lines)

        metadata = {
            "domain": "governance",
            "section": "questionnaire",
            "category": category.strip().lower(),
            "source_name": global_governance_context.get("source_name"),
            "project_name": global_governance_context.get("project_name")
        }

        chunks.append({
            "content": content,
            "metadata": metadata
        })

    return chunks

In [43]:
# testing

governance_chunks = build_governance_chunks(
    project_df,
    questionnaire_df,
    global_governance_context
)

print("Total governance chunks:", len(governance_chunks))
print("\nExample chunk:\n")
print(governance_chunks[2]["content"])
print("\nMetadata:\n")
print(governance_chunks[2]["metadata"])

Total governance chunks: 26

Example chunk:

Category: Audience

Who is the audience/department that has the appetite to analyze the data for the purpose of making business decisions that will help solve the business problem?  For example, business stakeholder groups, executive committees, front line management, etc.: Citizens Access business stakeholder groups, front line management

Metadata:

{'domain': 'governance', 'section': 'questionnaire', 'category': 'audience', 'source_name': 'FIS Profile', 'project_name': 'MBP 3.11 Upgrade'}


In [44]:
# for i, chunk in enumerate(governance_chunks):
#     print(f"\n==============================")
#     print(f"Chunk {i+1}")
#     print("Metadata:", chunk["metadata"])
#     print("\nContent:\n")
#     print(chunk["content"])

In [45]:
# count chunks per category 

from collections import Counter

categories = [
    chunk["metadata"].get("category")
    for chunk in governance_chunks
    if chunk["metadata"].get("category")
]

print(Counter(categories))

Counter({'additional sources': 1, 'audience': 1, 'availability': 1, 'business line': 1, 'change management': 1, 'connection info': 1, 'contacts': 1, 'dr in place': 1, 'data access': 1, 'data lake ad group': 1, 'data retention period': 1, 'documentation': 1, 'environments': 1, 'file/ db information': 1, 'financial significance': 1, 'masking': 1, 'party data': 1, 'products': 1, 'quality issues': 1, 'risks/assumptions/dependencies': 1, 'source description': 1, 'source name': 1, 'source owner approval': 1, 'sourcing strategy summary': 1, 'vendor information': 1})


In [46]:
# # chunking is done till now
# sheet 1 = 26 chunks 
# sheet 2 = 13 chunks 
# sheet 3 = 13 chunks  
# total   = 52 chunks 


# now combining  

In [47]:
# Each chunk has domain metadata
# goverance,file_information,technical mapping 
# even though we are stored togther,they are logical isolated using metadata filters 

In [48]:
# why we didnt link sheet1 to sheet 2 and sheet 3 
# sheet1=sourve level goveranance which is global 
# sheet2=file level 
# sheet 3=field level mapping 
# sheet applies to all 
# if we forced th elinkage : we would retrive noise,waste embedding space 

# what i did: keep goveranace data gloabl ,keep file level isolated ,keep tech levele stuctured.

In [49]:
# merging all chunks  
all_chunks = (
    governance_chunks +
    file_chunks +
    technical_chunks
)

print("Total combined chunks:", len(all_chunks))


Total combined chunks: 52


In [50]:
# chromadb does not store realtionship 
# it stores : embedddings, content ,metadata
# sheet3 and sheet2 are connected using standardize file key 
# so we build file tree not stored inchorma ,existis in memory during hcunk building 


In [51]:
# validating structure of each chunk :
def validate_chunks(chunks):

    for i, chunk in enumerate(chunks):

        if not chunk.get("content"):
            raise ValueError(f"Chunk {i} has empty content")

        if not chunk.get("metadata"):
            raise ValueError(f"Chunk {i} missing metadata")

        if "domain" not in chunk["metadata"]:
            raise ValueError(f"Chunk {i} missing domain")

    print("All chunks structurally valid.")

validate_chunks(all_chunks)


# Validate domain distribution  

from collections import Counter

domains = [c["metadata"]["domain"] for c in all_chunks]
print(Counter(domains))


# clenaing content 


All chunks structurally valid.
Counter({'governance': 26, 'file_information': 13, 'technical_mapping': 13})


In [52]:
import re

for i, chunk in enumerate(all_chunks):
    if re.search(r"\bnan\b", chunk["content"].lower()):
        print(f"\n---- Chunk {i} ----")
        print(chunk["metadata"])
        print("\nContent:\n")
        print(chunk["content"])

<!-- move to embedding + chroma insertion  -->
<!--sentence transformer (all MiniLM-L6-v2 ) -->
<!--Insert into chroma with maetadata  -->

In [53]:
# Embedding + Chroma insertion.
# Initialize SentenceTransformer (all-MiniLM-L6-v2)
# Generate embeddings
# Insert into Chroma with metadata


In [54]:
# Initialise the embedding Model  

from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

2026-03-04 10:31:08.392674: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [79]:
# initialize Persistent ChromaDB

import chromadb
from chromadb.config import Settings

chroma_client = chromadb.Client(
    Settings(
        persist_directory="./mbp_rag_db",
        is_persistent=True
    )
)

collection = chroma_client.get_or_create_collection(
    name="mbp_rag_collection"
)

In [80]:
# Inserting all chunks in chroma  

import uuid

documents = []
metadatas = []
ids = []
embeddings = []

for chunk in all_chunks:
    documents.append(chunk["content"])
    metadatas.append(chunk["metadata"])
    ids.append(str(uuid.uuid4()))

# Generate embeddings
embeddings = embedding_model.encode(
    documents,
    convert_to_numpy=True
).tolist()

# Insert into Chroma
collection.add(
    documents=documents,
    metadatas=metadatas,
    embeddings=embeddings,
    ids=ids
)

print("Inserted:", len(documents))

Inserted: 52


In [81]:
# Retrival without LLms 
def build_filter(domain=None, file_key=None, record_type=None):

    conditions = []

    if domain:
        conditions.append({"domain": domain})

    if file_key:
        conditions.append({"file_key": file_key})

    if record_type:
        conditions.append({"record_type": record_type})

    if not conditions:
        return None

    if len(conditions) == 1:
        return conditions[0]   # ← direct filter (no $and)

    return {"$and": conditions}
def retrieve_chunks(query, top_k=5, metadata_filter=None):

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    ).tolist()

    results = collection.query(
        query_embeddings=query_embedding,
        n_results=top_k,
        where=metadata_filter
    )

    return results

# testing

metadata_filter = build_filter(
    domain="file_information",
    file_key="mbp_accountextract_4300"
)

result = retrieve_chunks(
    "What is the frequency?",
    metadata_filter=metadata_filter
)

print("Number of results:", len(result["documents"][0]))

for doc, meta in zip(result["documents"][0], result["metadatas"][0]):
    print("\n----------------------------")
    print("Metadata:", meta)
    print("\nContent:\n")
    print(doc)

Number of results: 2

----------------------------
Metadata: {'frequency': 'Daily', 'load_type': 'Delta', 'domain': 'file_information', 'transmission': 'GIS', 'schema': 'DEPOSITS_TERRANOVA', 'file_key': 'mbp_accountextract_4300'}

Content:

File Name: MBP_AccountExtract_4300_YYYYMMDD.txt
Description: CAX Account File
Transmission Type: GIS
Frequency: Daily
Load Type: Delta
Autosys Start: 3 AM EST
Schema: DEPOSITS_TERRANOVA
Encrypted: N
Compressed: N
Producer Mailbox: DTECTB_FISGLOBAL1
Consumer Mailbox: DTI51171_ADVANCANAL02

----------------------------
Metadata: {'transmission': 'GIS', 'frequency': 'Daily', 'domain': 'file_information', 'schema': 'DEPOSITS_TERRANOVA', 'load_type': 'Delta', 'file_key': 'mbp_accountextract_4300'}

Content:

File Name: MBP_AccountExtract_4300_YYYYMMDD.txt
Description: CAX Account File
Transmission Type: GIS
Frequency: Daily
Load Type: Delta
Autosys Start: 3 AM EST
Schema: DEPOSITS_TERRANOVA
Encrypted: N
Compressed: N
Producer Mailbox: DTECTB_FISGLOBAL1
C

In [82]:
# question: what is the tranmission type of mbp account extrcat?  
metadata_filter = build_filter(
    domain="file_information",
    file_key="mbp_accountextract_4300"
)

result = retrieve_chunks(
    "What is the transmission type?",
    metadata_filter=metadata_filter
)

for doc, meta in zip(result["documents"][0], result["metadatas"][0]):
    print(meta)
    print(doc)

{'schema': 'DEPOSITS_TERRANOVA', 'file_key': 'mbp_accountextract_4300', 'domain': 'file_information', 'frequency': 'Daily', 'load_type': 'Delta', 'transmission': 'GIS'}
File Name: MBP_AccountExtract_4300_YYYYMMDD.txt
Description: CAX Account File
Transmission Type: GIS
Frequency: Daily
Load Type: Delta
Autosys Start: 3 AM EST
Schema: DEPOSITS_TERRANOVA
Encrypted: N
Compressed: N
Producer Mailbox: DTECTB_FISGLOBAL1
Consumer Mailbox: DTI51171_ADVANCANAL02
{'domain': 'file_information', 'transmission': 'GIS', 'load_type': 'Delta', 'schema': 'DEPOSITS_TERRANOVA', 'file_key': 'mbp_accountextract_4300', 'frequency': 'Daily'}
File Name: MBP_AccountExtract_4300_YYYYMMDD.txt
Description: CAX Account File
Transmission Type: GIS
Frequency: Daily
Load Type: Delta
Autosys Start: 3 AM EST
Schema: DEPOSITS_TERRANOVA
Encrypted: N
Compressed: N
Producer Mailbox: DTECTB_FISGLOBAL1
Consumer Mailbox: DTI51171_ADVANCANAL02


In [59]:
# sheet3 mapping 
metadata_filter = build_filter(
    domain="technical_mapping",
    file_key="mbp_accountextract_4300",
    record_type="9000"
)

result = retrieve_chunks(
    "What fields are in this record type?",
    metadata_filter=metadata_filter
)

for doc, meta in zip(result["documents"][0], result["metadatas"][0]):
    print(meta)
    print(doc) 

{'schema': 'CL_DEPOSITS_TERRANOVA', 'table': 'APD_AR_TDL_DP', 'load_type': 'Delta', 'transmission': 'GIS', 'domain': 'technical_mapping', 'record_type': '9000', 'frequency': 'Daily', 'file_key': 'mbp_accountextract_4300'}
File: MBP_AccountExtract_4300_YYYYMMDD.txt
Record Type: 9000
Target Schema: CL_DEPOSITS_TERRANOVA
Target Table: APD_AR_TDL_DP
Frequency: Daily
Transmission: GIS
Load Type: Delta

Fields:
- ARID_ITEM_IBAN (STRING 40) → ARID_ITEM_IBAN (STRING None)
- REC_TYP (NUMBER 4) → REC_TYP (NUMBER None)
- ARF_IDFR (NUMBER 15) → ARF_IDFR (NUMBER None)
- ARID_ITEM (STRING 40) → ARID_ITEM (STRING None)
- ARID_ITEM_MICR (STRING 40) → ARID_ITEM_MICR (STRING None)


In [60]:
# testing sheet 1 
metadata_filter = build_filter(
    domain="governance"
)

result = retrieve_chunks(
    "Who is the vendor?",
    metadata_filter=metadata_filter
)

for doc, meta in zip(result["documents"][0], result["metadatas"][0]):
    print(meta)
    print(doc)

{'domain': 'governance', 'category': 'vendor information', 'source_name': 'FIS Profile', 'section': 'questionnaire', 'project_name': 'MBP 3.11 Upgrade'}
Category: Vendor Information

Company: FIS
Software Name: Profile
Contacts: Tapsie Iyer - Vendor contact
Contact Preferences (Who to raise questions through): Tom Williamson
{'project_name': 'MBP 3.11 Upgrade', 'category': 'contacts', 'domain': 'governance', 'section': 'questionnaire', 'source_name': 'FIS Profile'}
Category: Contacts

Application Suite manager: Tom Williamson
Application manager: Michael Collins
Application business contact: John Rosenfeld
Source Data Steward: TBD - Business to confirm
Source Trustee: TBD - Business to confirm
Support Distribution List: FIS.Profile.Outsourcing.ECC.Operations@fisglobal.com; 800-322-2741 option 2, option 4, option 4.
Other Identified SMEs: Tapsie Iyer - Vendor contact
{'source_name': 'FIS Profile', 'domain': 'governance', 'category': 'source name', 'project_name': 'MBP 3.11 Upgrade', 'se

In [61]:
metadata_filter = build_filter(domain="governance")

result = retrieve_chunks(
    "Who is the vendor?",
    top_k=1,
    metadata_filter=metadata_filter
)

print(result["metadatas"][0][0])
print(result["documents"][0][0])

{'domain': 'governance', 'project_name': 'MBP 3.11 Upgrade', 'category': 'vendor information', 'section': 'questionnaire', 'source_name': 'FIS Profile'}
Category: Vendor Information

Company: FIS
Software Name: Profile
Contacts: Tapsie Iyer - Vendor contact
Contact Preferences (Who to raise questions through): Tom Williamson


In [62]:
# routing layer automation 
# we were manually doing :
# domain filter
# file_key filter
# record_type filter


In [63]:
# A query router function that automatically:
# Detects domain
# governance
# file_information
# technical_mapping
# Extracts file name if present
# converts to standardized_file_key
# Extracts record_type if present
# Builds correct metadata filter
# Calls retrieve_chunks()
# Returns clean context ready for LLM

In [83]:
# build router model  for retrival efficiently 


import re
available_file_keys=list(file_tree.keys())

# -----------------------------
# FILE KEY DETECTOR
# -----------------------------
def detect_file_key(query, file_information_df):

    query_clean = re.sub(r"[^a-zA-Z0-9 ]", "", query.lower())

    best_match = None
    best_score = 0

    for _, row in file_information_df.iterrows():

        file_name = str(row["file_name"]).lower()
        file_name_clean = re.sub(r"[^a-zA-Z0-9 ]", "", file_name)

        # simple overlap score
        score = sum(
            1 for word in query_clean.split()
            if word in file_name_clean
        )

        if score > best_score:
            best_score = score
            best_match = row["standardized_file_key"]

    if best_score >= 2:   # threshold to avoid random matches
        return best_match

    return None


# -----------------------------
# RECORD TYPE DETECTOR
# -----------------------------
def detect_record_type(query):
    match = re.search(r"\b\d{4}\b", query)
    if match:
        return match.group()
    return None


# -----------------------------
# DOMAIN DETECTOR
# -----------------------------
def detect_domain(query):

    q = query.lower()

    governance_keywords = [
        "vendor", "risk", "contact", "retention",
        "business line", "security", "approval",
        "availability", "environment"
    ]

    technical_keywords = [
        "field", "column", "record type", "datatype",
        "table", "schema"
    ]

    file_keywords = [
        "frequency", "transmission", "encrypted",
        "compressed", "autosys", "load type"
    ]

    if any(word in q for word in technical_keywords):
        return "technical_mapping"

    if any(word in q for word in file_keywords):
        return "file_information"

    if any(word in q for word in governance_keywords):
        return "governance"

    return "governance"  # safe fallback


# -----------------------------
# MAIN ROUTER
# -----------------------------
def route_query(query):

    domain = detect_domain(query)

    file_key = detect_file_key(query, file_information_df)

    record_type = detect_record_type(query)

    metadata_filter = build_filter(
        domain=domain,
        file_key=file_key,
        record_type=record_type
    )

    print("Detected domain:", domain)
    print("Detected file_key:", file_key)
    print("Detected record_type:", record_type)

    results = retrieve_chunks(
        query,
        metadata_filter=metadata_filter,
        top_k=3
    )

    return results



In [84]:
result = route_query(
    "What is the frequency of MBP Account Extract?"
)

for doc, meta in zip(result["documents"][0], result["metadatas"][0]):
    print("\nMetadata:", meta)
    print("Content:\n", doc)

Detected domain: file_information
Detected file_key: mbp_accountextract_4300
Detected record_type: None

Metadata: {'frequency': 'Daily', 'load_type': 'Delta', 'transmission': 'GIS', 'file_key': 'mbp_accountextract_4300', 'domain': 'file_information', 'schema': 'DEPOSITS_TERRANOVA'}
Content:
 File Name: MBP_AccountExtract_4300_YYYYMMDD.txt
Description: CAX Account File
Transmission Type: GIS
Frequency: Daily
Load Type: Delta
Autosys Start: 3 AM EST
Schema: DEPOSITS_TERRANOVA
Encrypted: N
Compressed: N
Producer Mailbox: DTECTB_FISGLOBAL1
Consumer Mailbox: DTI51171_ADVANCANAL02

Metadata: {'domain': 'file_information', 'transmission': 'GIS', 'schema': 'DEPOSITS_TERRANOVA', 'frequency': 'Daily', 'load_type': 'Delta', 'file_key': 'mbp_accountextract_4300'}
Content:
 File Name: MBP_AccountExtract_4300_YYYYMMDD.txt
Description: CAX Account File
Transmission Type: GIS
Frequency: Daily
Load Type: Delta
Autosys Start: 3 AM EST
Schema: DEPOSITS_TERRANOVA
Encrypted: N
Compressed: N
Producer Mailb

In [85]:
result = route_query(
    "What fields are in record type 9000 of MBP Account Extract?"
)

for doc, meta in zip(result["documents"][0], result["metadatas"][0]):
    print("\nMetadata:", meta)
    print("Content:\n", doc)


Detected domain: technical_mapping
Detected file_key: mbp_accountextract_4300
Detected record_type: 9000

Metadata: {'schema': 'CL_DEPOSITS_TERRANOVA', 'frequency': 'Daily', 'load_type': 'Delta', 'transmission': 'GIS', 'record_type': '9000', 'file_key': 'mbp_accountextract_4300', 'domain': 'technical_mapping', 'table': 'APD_AR_TDL_DP'}
Content:
 File: MBP_AccountExtract_4300_YYYYMMDD.txt
Record Type: 9000
Target Schema: CL_DEPOSITS_TERRANOVA
Target Table: APD_AR_TDL_DP
Frequency: Daily
Transmission: GIS
Load Type: Delta

Fields:
- ARID_ITEM_IBAN (STRING 40) → ARID_ITEM_IBAN (STRING None)
- REC_TYP (NUMBER 4) → REC_TYP (NUMBER None)
- ARF_IDFR (NUMBER 15) → ARF_IDFR (NUMBER None)
- ARID_ITEM (STRING 40) → ARID_ITEM (STRING None)
- ARID_ITEM_MICR (STRING 40) → ARID_ITEM_MICR (STRING None)

Metadata: {'domain': 'technical_mapping', 'load_type': 'Delta', 'file_key': 'mbp_accountextract_4300', 'frequency': 'Daily', 'table': 'APD_AR_TDL_DP', 'schema': 'CL_DEPOSITS_TERRANOVA', 'transmission':

In [86]:
result = route_query(
    "Who is the vendor?"
)

for doc, meta in zip(result["documents"][0], result["metadatas"][0]):
    print("\nMetadata:", meta)
    print("Content:\n", doc)

Detected domain: governance
Detected file_key: None
Detected record_type: None

Metadata: {'source_name': 'FIS Profile', 'domain': 'governance', 'project_name': 'MBP 3.11 Upgrade', 'category': 'vendor information', 'section': 'questionnaire'}
Content:
 Category: Vendor Information

Company: FIS
Software Name: Profile
Contacts: Tapsie Iyer - Vendor contact
Contact Preferences (Who to raise questions through): Tom Williamson

Metadata: {'project_name': 'MBP 3.11 Upgrade', 'category': 'vendor information', 'section': 'questionnaire', 'domain': 'governance', 'source_name': 'FIS Profile'}
Content:
 Category: Vendor Information

Company: FIS
Software Name: Profile
Contacts: Tapsie Iyer - Vendor contact
Contact Preferences (Who to raise questions through): Tom Williamson

Metadata: {'section': 'questionnaire', 'source_name': 'FIS Profile', 'domain': 'governance', 'project_name': 'MBP 3.11 Upgrade', 'category': 'contacts'}
Content:
 Category: Contacts

Application Suite manager: Tom Williamson

In [87]:
# using llm now 
# final step 


In [88]:
# prompt template 

SYSTEM_PROMPT = """
You are a Data Lake Governance and Technical Metadata Specialist.

You must answer strictly using ONLY the provided context.
Do NOT use outside knowledge.
Do NOT assume anything not explicitly present.
Do NOT fabricate missing details.

If the answer is not found in the provided context, respond professionally with:
"The requested information is not available in the provided Data Lake documentation."

Provide clear, concise, and professional answers.
If multiple relevant details exist, structure them clearly.
"""

In [89]:
# context builder 
def build_context(results):

    docs = results["documents"][0]

    combined_context = "\n\n---\n\n".join(docs)

    return combined_context  



In [90]:
# bedrock generation function 
import json
import boto3

bedrock_runtime = boto3.client(
    service_name="bedrock-runtime",
    region_name="ap-south-1"
)

MODEL_ID = "anthropic.claude-3-haiku-20240307-v1:0"

SYSTEM_PROMPT = """
You are a Data Lake Governance and Technical Metadata Specialist.

You must answer strictly using ONLY the provided context.
Do NOT use outside knowledge.
Do NOT assume anything not explicitly present.
Do NOT fabricate missing details.

If the answer is not found in the provided context, respond professionally with:
"The requested information is not available in the provided Data Lake documentation."

Provide clear, concise, and professional answers.
If multiple relevant details exist, structure them clearly.
"""

def generate_answer(query, retrieved_results):

    context = build_context(retrieved_results)

    full_prompt = f"""
{SYSTEM_PROMPT}

CONTEXT:
{context}

USER QUESTION:
{query}

Answer:
"""

    body = {
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": 500,
        "temperature": 0,
        "messages": [
            {
                "role": "user",
                "content": full_prompt
            }
        ]
    }

    response = bedrock_runtime.invoke_model(
        modelId=MODEL_ID,
        body=json.dumps(body)
    )

    response_body = json.loads(response["body"].read())

    return response_body["content"][0]["text"]


In [92]:
# user interaction 
def ask_rag_system(query):

    # Step 1 — Route & Retrieve
    retrieved = route_query(query)

    # Step 2 — Generate Answer
    answer = generate_answer(query, retrieved)

    return answer

In [93]:
response = ask_rag_system(
    "What is the frequency of MBP Account Extract?"
)

print(response)

Detected domain: file_information
Detected file_key: mbp_accountextract_4300
Detected record_type: None
The frequency of the MBP Account Extract file is Daily, as stated in the provided context.


In [94]:
response = ask_rag_system(
    "Who is the vendor?"
)

print(response)

Detected domain: governance
Detected file_key: None
Detected record_type: None
The vendor is FIS, as stated in the provided context.


In [95]:
response = ask_rag_system(
    "What is the CEO salary?"
)

print(response)

Detected domain: governance
Detected file_key: None
Detected record_type: None
The requested information is not available in the provided Data Lake documentation. The context provided does not contain any information about the CEO salary. As a Data Lake Governance and Technical Metadata Specialist, I do not have access to that level of detail about the FIS Profile source. The available information is limited to the source name, technical and business aliases, and contact details for the application and data management personnel.


In [97]:
import ipywidgets as widgets
from IPython.display import display, clear_output



sample_box = widgets.Textarea(
    # value="\n".join(sample_questions),
    description="Sample Questions",
    layout=widgets.Layout(width="100%", height="200px")
)

question_input = widgets.Textarea(
    placeholder="Enter your question here...",
    description="Your Question",
    layout=widgets.Layout(width="100%", height="100px")
)

ask_button = widgets.Button(
    description="Ask",
    button_style="primary"
)

output_box = widgets.Output()

def on_ask_clicked(b):
    with output_box:
        clear_output()
        query = question_input.value.strip()
        if not query:
            print("Please enter a question.")
            return
        response = ask_rag_system(query)
        print(response)

ask_button.on_click(on_ask_clicked)

# display(sample_box)
display(question_input)
display(ask_button)
display(output_box)

Textarea(value='', description='Your Question', layout=Layout(height='100px', width='100%'), placeholder='Ente…

Button(button_style='primary', description='Ask', style=ButtonStyle())

Output()

In [77]:
# sample_questions = [
#     "What is the frequency of MBP Account Extract?",
#     "What is the transmission type of MBP Transaction Extract?",
#     "What is the autosys start time of MBP Account Extract?",
#     "Is MBP Account Extract encrypted?",
#     "What fields are in record type 9000 of MBP Account Extract?",
#     "Which schema does MBP Account Extract load into?",
#     "Who is the vendor?",
#     "What is the data retention period?",
#     "What environments exist for this source?",
#     "What is the sourcing strategy?"
# ]